# Initialize 

In [ ]:
from pathlib import Path
import numpy as np
import yaml
import xarray as xr
import pymc as pm
import matplotlib.pyplot as plt
from pymc.model.transform.optimization import freeze_dims_and_data
import arviz as az
from tqdm.notebook import tqdm
import model_chain_inference as mci

In [ ]:
plt.rcParams["figure.figsize"] = [12, 5]
%config InlineBackend.figure_format = 'retina'

# Experiment settings

In [ ]:
scenario = "performance_1995_2025"
perspectives = ["prospective", "retrospective"]

RANDOM_SEED = 1234
n_samples = 1000
n_chains = 8

path = Path("../data")
grid_data_file = path / "grid_data_flat.h5"
model_specs_file = path / f"model_specs-{scenario}.yaml"

# Data prep

## Prepare inference datasets

An inference dataset prepares the data of a data scenario into a compact format that is ready for inference.

In [ ]:
def get_inference_data(path, scenario, perspective, model):
    ids = xr.open_dataset(
        path / f"model_data-{scenario}-{perspective}-{model}.h5",
        decode_timedelta=False,
        engine="h5netcdf",
    )
    if "bernstein_basis" in ids.dims:
        ids = ids.set_xindex(["bernstein_degree", "bernstein_index"])
    return ids

In [ ]:
def get_idata_path(scenario, perspective, model):
    return path / f"model_calibration-{scenario}-{perspective}-{model}.h5"

# Model definitions

In [ ]:
settings_dsm = {
    "default": {
        "interpolation_dims": [
            "rmax",
            "sigma",
            "hs_exp",
            "hs_exp_alt",
            "M_plastic",
        ],
    },
}

In [ ]:
settings_rate = {
    "default": {
        "mixing_dims": ["bernstein_index"],
        "theta0": {
            "dist": "Normal",
            "mu": -17.0,
            "sigma": 5.0,
        },
        "theta1": {
            "dist": "Exponential",
            "scale": 5.0,
            "dims": ["bernstein_index"],
        },
    },
}

In [ ]:
settings_etas = {
    "default": None,
    "Kac": {
        "interpolation_dims": [
            "etas_d",
            "etas_q",
        ],
        "etas_c": {"dist": "LogNormal", "mu": 0.0, "sigma": 1.0},
        "etas_p": 1.35,
        "etas_K": {"dist": "Exponential", "scale": 0.1},
        "etas_a": {"dist": "Uniform", "lower": 0.0, "upper": 2.0},
    },
}

In [ ]:
settings_size = {"default": None}

In [ ]:
model_specs = {
    "ETS_bs4_etasKac": {
        "data_id": "ETS_bs",
        "etas_model_id": "Kac",
        "sel": {"thickness_origin": "grid", "bernstein_degree": 4},
    },
    "EVTS_bs4_etasKac": {
        "data_id": "EVTS_bs",
        "etas_model_id": "Kac",
        "sel": {"thickness_origin": "grid", "bernstein_degree": 4},
    },
    "ETS_rmax_etasKac": {
        "data_id": "ETS_rmax",
        "etas_model_id": "Kac",
        "sel": {"thickness_origin": "grid"},
    },
    "EVTS_rmax_etasKac": {
        "data_id": "EVTS_rmax",
        "etas_model_id": "Kac",
        "sel": {"thickness_origin": "grid"},
    },
}

In [ ]:
with open(model_specs_file, "w") as f:
    yaml.dump(model_specs, f, default_flow_style=False)

In [ ]:
model_collection = {}
for k, pymc_model in tqdm(model_specs.items()):
    for perspective in perspectives:
        name = f"{scenario}-{perspective}-{k}"
        tqdm.write(f"creating model: {name}")

        specs_local = pymc_model.copy()
        sel = specs_local.pop("sel", {})
        isel = specs_local.pop("isel", {})
        data_id = specs_local.pop("data_id")
        data_local = (
            get_inference_data(path, scenario, perspective, data_id).sel(sel).isel(isel)
        )

        rate_model_id = specs_local.pop("rate_model_id", "default")
        size_model_id = specs_local.pop("size_model_id", "default")
        dsm_model_id = specs_local.pop("dsm_model_id", "default")
        etas_model_id = specs_local.pop("etas_model_id", "default")
        settings_local = {
            "rate_parameter_data": settings_rate[rate_model_id],
            "size_parameter_data": settings_size[size_model_id],
            "dsm_parameter_data": settings_dsm[dsm_model_id],
            "etas_parameter_data": settings_etas[etas_model_id],
        }

        model_collection[name] = mci.generate_and_test_model(
            mci.generate_ts_etf_etas_model,
            data=data_local,
            settings=settings_local,
        )

# Inference

In [ ]:
result_collection = {}
for k, specs in tqdm(model_collection.items()):
    tqdm.write(f"\n{k}")
    frozen_model = freeze_dims_and_data(specs)
    with frozen_model:
        model = k
        name = f"{scenario}-{perspective}-{model}"
        if len(specs.discrete_value_vars) == 0:
            nuts_sampler = "blackjax"
        else:
            nuts_sampler = "pymc"
        if name in result_collection:
            tqdm.write(f"model {name} already sampled - skipping")
            continue
        tqdm.write(f"nuts_sampler: {nuts_sampler}")
        idata_path = get_idata_path(scenario, perspective, name)
        if idata_path.exists():
            tqdm.write(f"file {idata_path.as_posix()} exists - remove to overwrite")
            result_collection[name] = az.from_netcdf(idata_path)
            continue
        else:
            rng = np.random.default_rng(RANDOM_SEED)
            try:
                idata = pm.sample(
                    draws=n_samples,
                    chains=n_chains,
                    cores=n_chains,
                    target_accept=0.9,
                    idata_kwargs={"log_likelihood": True},
                    random_seed=rng,
                    nuts_sampler=nuts_sampler,
                )
                display(az.summary(idata))
                result_collection[name] = idata
                idata.to_netcdf(idata_path)
            except Exception as e:
                tqdm.write(f"error: {e}\n")

In [ ]:
azc = az.compare(result_collection, ic="loo", var_name="XT")
az.plot_compare(azc, insample_dev=True, plot_ic_diff=True)
azc